# B4 · Does task relevance improve transfer?
Compare a text-only DeBERTa control, a paired cross-encoder, the same pair with explicit relevance, and a BIPIA-only training ablation. The full experiment runs each configuration with three seeds.

**First run.** Choose a GPU under **Runtime → Change runtime type**. Have the accompanying `b4-colab-source.zip` ready for the first upload. The notebook saves it and the experiment outputs under your Google Drive. For DVC data access, add your team's read-only R2 keys to Colab Secrets under `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY`. The upload data mode below works with an already-prepared local dataset and context sidecar.

Start with **smoke** and run all cells. Then change the mode to **full** and run all again. To run the optional learning-rate search, set `INCLUDE_SWEEP = True` before running full. Keep enough Drive space for twelve saved models and the active checkpoint. Use a new study name when changing source or configuration.

In [ ]:
MODE = "smoke"  # @param ["smoke", "full", "sweep"]
DATA_MODE = "dvc"  # @param ["dvc", "upload"]
INCLUDE_SWEEP = False  # @param {type:"boolean"}
STUDY = "b4-relevance-v1"  # @param {type:"string"}

## Set up the experiment
The code archive contains the experiment source and dependency lock file. Build it from a local checkout with `python scripts/package_b4_colab.py`. The upload chooser appears once per study. After a disconnect, rerun the notebook using the same study name.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import re
import shutil
import subprocess
import sys
from google.colab import drive, files

assert re.fullmatch(r"[A-Za-z0-9_-]+", STUDY), "Use a simple study folder name."
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/cs7267-b4") / STUDY
ROOT.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/b4-work") / STUDY
WORK.mkdir(parents=True, exist_ok=True)
archive = ROOT / "b4-colab-source.zip"
if not archive.is_file():
    print("Upload b4-colab-source.zip")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select the single B4 code archive.")
    archive.write_bytes(next(iter(uploaded.values())))
    del uploaded
with ZipFile(archive) as source:
    for entry in source.namelist():
        if not (WORK / entry).resolve().is_relative_to(WORK.resolve()):
            raise ValueError("Unexpected archive path")
    if "src/models/b4_experiments.py" not in source.namelist():
        raise ValueError("Expected the code archive made by package_b4_colab.py")
    source.extractall(WORK)
os.chdir(WORK)
env = {**os.environ, "PYTHONPATH": os.pathsep.join(str(WORK / p) for p in ["src/data", "src/eval", "src/models"])}
def run(*args, custom_env=None):
    subprocess.run([str(x) for x in args], cwd=WORK, env=env if custom_env is None else custom_env, check=True)

if not shutil.which("uv"):
    run(sys.executable, "-m", "pip", "install", "-q", "uv")
if not (WORK / ".git").exists():
    run("git", "init", "-q")
run("uv", "sync", "--python", "3.11", "--frozen", "--group", "deberta", "--group", "dev")
PY = WORK / ".venv/bin/python"
run(PY, "-c", "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))")
print("Study folder", ROOT)

## Prepare the paired data
The notebook joins original row IDs to independently supplied questions or task catalogues. For the upload option, select all three local files together: `dataset.parquet`, `contexts.parquet` and `contexts.coverage.csv`. For DVC, credentials go only to the download process.

In [ ]:
import pandas as pd
DATA = WORK / "data/processed/dataset.parquet"
CTX = ROOT / "inputs/contexts.parquet"
COVERAGE = CTX.with_suffix(".coverage.csv")
CTX.parent.mkdir(parents=True, exist_ok=True)
if DATA_MODE == "dvc":
    from google.colab import userdata
    targets = ["data/processed/dataset.parquet"]
    if not CTX.exists() or not COVERAGE.exists():
        targets.extend(["data/raw", "data/generated/bipia/qa", "data/generated/bipia/abstract"])
    if not DATA.exists() or len(targets) > 1:
        run(WORK / ".venv/bin/dvc", "pull", *targets, custom_env={
            **env, "AWS_ACCESS_KEY_ID": userdata.get("R2_ACCESS_KEY_ID"),
            "AWS_SECRET_ACCESS_KEY": userdata.get("R2_SECRET_ACCESS_KEY")})
    if not CTX.exists() or not COVERAGE.exists():
        run(PY, "src/data/b4_data.py", "--data", DATA, "--out", CTX)
else:
    DATA = ROOT / "inputs/dataset.parquet"
    required = [DATA, CTX, COVERAGE]
    if not all(p.is_file() for p in required):
        received = files.upload()
        for p in required:
            if p.name in received:
                p.write_bytes(received[p.name])
        del received
    if not all(p.is_file() for p in required):
        raise ValueError("Expected dataset.parquet, contexts.parquet and contexts.coverage.csv")
coverage = pd.read_csv(COVERAGE)
display(coverage.groupby(["source", "label"])[["total", "supported", "excluded"]].sum())

## Measure the relevance features on validation
Fit a logistic regression on just the two lexical features. This CPU control measures their signal before fine-tuning. The next experiment measures their added value beside DeBERTa.

In [ ]:
import json
probe = ROOT / 'relevance_probe_validation.json'
run(PY, 'src/models/b4_probe.py', '--data', DATA, '--contexts', CTX, '--out', probe)
display(pd.DataFrame([json.loads(probe.read_text())])[['n_train', 'n_val', 'val_auroc', 'bootstrap_95', 'val_tpr', 'val_fpr']])

## Run the experiments
**Smoke** trains one small run and scores validation. **Sweep** compares two learning rates on validation. **Full** trains the 12 reported runs, then scores the familiar test split and the supported MCPTox and MSB rows. A full run repeats the sweep automatically if enabled, reusing finished training. Each epoch is checkpointed while its run is active. Model, score, and report files are written to Drive.

In [ ]:
import json
# Use the project interpreter for YAML and all experiment code.
cfg = ROOT / "b4.yaml"
run(PY, "-c", """import sys,yaml
from pathlib import Path
c=yaml.safe_load(Path('configs/b4.yaml').read_text())
c['search']['enabled']=sys.argv[2]=='True'
Path(sys.argv[1]).write_text(yaml.safe_dump(c, sort_keys=False))
""", cfg, str(INCLUDE_SWEEP))
OUT = ROOT / ("experiments-sweep" if INCLUDE_SWEEP else "experiments-fixed")
run(PY, "src/models/b4_experiments.py", "--config", cfg, "--out", OUT, "--mode", MODE,
    "--data", DATA, "--contexts", CTX, "--device", "cuda")
print("Outputs", OUT)

## Inspect results
The feature comparison is `pair_only → relevance` at the same seed. Positive AUROC or matched-5%-FPR detection differences indicate improvement for that seed. The two tool benchmarks use server task catalogues as a proxy for the runtime request. Report coverage alongside their scores.

In [ ]:
from IPython.display import Image, Markdown, display
if MODE == "full":
    R = OUT / "reports"
    display(Markdown((R / "summary.md").read_text()), Image(filename=str(R / "auroc.png")))
    effects = pd.read_csv(R / "ablation_deltas.csv")
    display(effects.query("slice == 'all' and comparison == 'pair_only -> relevance' and metric in ['auroc', 'tpr_at_matched_fpr']").pivot_table(
        index=["eval_set", "metric"], columns="seed", values="delta"))
    for name in ("reference_length", "reference_tfidf"):
        display(pd.read_csv(R / (name + ".csv")).query("slice == 'all'")[["detector", "eval_set", "auroc", "tpr_at_matched_fpr"]])
elif MODE == "sweep":
    display(pd.read_csv(OUT / "sweep/trials.csv"))
else:
    smoke = next((OUT / "smoke").glob("*/model/run.json"))
    display(pd.DataFrame(json.loads(smoke.read_text())["history"]))